# 03-8 클래스와 데이터클래스 실습

클래스·인스턴스, 메서드 바인딩, 클래스 변수, 상태 전이, property, dataclass, 상속·합성을 순서대로 실습합니다. 마지막에는 03-7 이벤트 딕셔너리를 검증된 객체와 배치로 변환합니다.

## 1. 클래스와 서로 다른 인스턴스

In [ ]:
class Book:
    pass

first_book = Book()
second_book = Book()
book_alias = first_book

assert isinstance(first_book, Book)
assert type(first_book) is Book
assert first_book is not second_book
assert book_alias is first_book
print(type(first_book).__name__, first_book is second_book, book_alias is first_book)

## 2. `__init__`, `self`, 결합된 메서드

In [ ]:
class Book:
    def __init__(self, title, author):
        if not isinstance(title, str):
            raise TypeError("title은 문자열이어야 합니다")
        if not title.strip():
            raise ValueError("title은 비어 있을 수 없습니다")
        self.title = title.strip()
        self.author = author.strip()

    def describe(self):
        return f"{self.title} / {self.author}"

book = Book(" Python 기초 ", "교육팀")
bound_describe = book.describe

assert book.title == "Python 기초"
assert book.describe() == "Python 기초 / 교육팀"
assert Book.describe(book) == book.describe()
assert bound_describe.__self__ is book
print(bound_describe())

## 3. 인스턴스 변수와 클래스 변수

In [ ]:
class Task:
    category = "learning"

    def __init__(self, title):
        self.title = title
        self.done = False
        self.tags = []

first_task = Task("교안 읽기")
second_task = Task("문제 풀기")
first_task.done = True
first_task.tags.append("important")
first_task.category = "review"

assert first_task.done is True and second_task.done is False
assert first_task.tags == ["important"] and second_task.tags == []
assert first_task.tags is not second_task.tags
assert first_task.category == "review"
assert second_task.category == Task.category == "learning"
print(first_task.__dict__, second_task.__dict__)

In [ ]:
class BadTask:
    tags = []

bad_first = BadTask()
bad_second = BadTask()
bad_first.tags.append("shared")

assert bad_first.tags is bad_second.tags
assert bad_second.tags == ["shared"]
print("공유 리스트 문제를 재현했습니다:", bad_second.tags)

## 4. 상태 전이와 예상 예외 검증

In [ ]:
def expect_exception(expected_type, function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except expected_type as exc:
        return exc
    except Exception as exc:
        raise AssertionError(
            f"{expected_type.__name__} 대신 {type(exc).__name__} 발생"
        ) from exc
    raise AssertionError(f"{expected_type.__name__}이 발생하지 않음")

class Ticket:
    def __init__(self, title):
        if not title.strip():
            raise ValueError("title은 비어 있을 수 없습니다")
        self.title = title.strip()
        self.closed = False

    def close(self):
        if self.closed:
            raise ValueError("이미 닫힌 티켓입니다")
        self.closed = True

ticket = Ticket("문서 검토")
ticket.close()
duplicate_close = expect_exception(ValueError, ticket.close)
empty_title = expect_exception(ValueError, Ticket, "   ")

assert ticket.closed is True
assert "이미 닫힌" in str(duplicate_close)
assert "비어" in str(empty_title)
print(duplicate_close)

## 5. 인스턴스·클래스·정적 메서드

In [ ]:
class Endpoint:
    def __init__(self, host, port):
        if not host.strip():
            raise ValueError("host는 비어 있을 수 없습니다")
        if not self.is_valid_port(port):
            raise ValueError("port는 1~65535 정수여야 합니다")
        self.host = host.strip()
        self.port = port

    @classmethod
    def from_text(cls, text):
        host, port_text = text.rsplit(":", 1)
        return cls(host, int(port_text))

    @staticmethod
    def is_valid_port(port):
        return type(port) is int and 1 <= port <= 65535

    def as_text(self):
        return f"{self.host}:{self.port}"

endpoint = Endpoint.from_text("example.test:443")
assert endpoint.as_text() == "example.test:443"
assert Endpoint.is_valid_port(443) is True
assert Endpoint.is_valid_port(True) is False
print(endpoint.as_text())

## 6. property로 속성 규칙 유지

In [ ]:
class Rating:
    def __init__(self, score):
        self._score = 1
        self.score = score

    @property
    def score(self):
        return self._score

    @score.setter
    def score(self, value):
        if type(value) is not int:
            raise TypeError("score는 정수여야 합니다")
        if not 1 <= value <= 5:
            raise ValueError("score는 1~5 범위여야 합니다")
        self._score = value

rating = Rating(3)
rating.score = 5
assert rating.score == 5
expect_exception(ValueError, setattr, rating, "score", 6)
expect_exception(TypeError, setattr, rating, "score", True)
assert rating.score == 5
print(rating.score)

## 7. 특수 메서드와 값 동등성

In [ ]:
class ManualBook:
    def __init__(self, title, author):
        self.title = title
        self.author = author

    def __repr__(self):
        return f"ManualBook(title={self.title!r}, author={self.author!r})"

    def __str__(self):
        return f"{self.title} / {self.author}"

    def __eq__(self, other):
        if not isinstance(other, ManualBook):
            return NotImplemented
        return (self.title, self.author) == (other.title, other.author)

manual_first = ManualBook("Python", "교육팀")
manual_second = ManualBook("Python", "교육팀")
assert manual_first == manual_second
assert manual_first is not manual_second
assert str(manual_first) == "Python / 교육팀"
assert repr(manual_first).startswith("ManualBook(")
print(repr(manual_first), str(manual_first))

## 8. dataclass와 `default_factory`

In [ ]:
from dataclasses import FrozenInstanceError, dataclass, field

@dataclass
class Finding:
    title: str
    severity: str
    tags: list[str] = field(default_factory=list)

    def __post_init__(self):
        if not isinstance(self.title, str):
            raise TypeError("title은 문자열이어야 합니다")
        self.title = self.title.strip()
        if not self.title:
            raise ValueError("title은 비어 있을 수 없습니다")
        self.severity = self.severity.strip().upper()
        if self.severity not in {"LOW", "MEDIUM", "HIGH"}:
            raise ValueError("지원하지 않는 severity입니다")

first_finding = Finding(" 입력 검증 ", "high")
second_finding = Finding("입력 검증", "HIGH")
first_finding.tags.append("validation")

assert first_finding.title == "입력 검증"
assert first_finding.severity == "HIGH"
assert first_finding == Finding("입력 검증", "HIGH", ["validation"])
assert second_finding.tags == []
assert first_finding.tags is not second_finding.tags
expect_exception(ValueError, Finding, "title", "critical")
print(first_finding)

## 9. frozen은 깊은 불변성이 아니다

In [ ]:
@dataclass(frozen=True)
class Coordinate:
    x: int
    y: int

point = Coordinate(10, 20)
expect_exception(FrozenInstanceError, setattr, point, "x", 99)
assert point.x == 10

@dataclass(frozen=True)
class FrozenReport:
    title: str
    rows: list[str] = field(default_factory=list)

frozen_report = FrozenReport("daily")
frozen_report.rows.append("row-1")
assert frozen_report.rows == ["row-1"]
print("필드 재대입은 거부되지만 내부 리스트는 변경됨:", frozen_report)

## 10. 상속과 합성

In [ ]:
class NamedItem:
    def __init__(self, name):
        self.name = name

    def label(self):
        return self.name

class NamedBook(NamedItem):
    def __init__(self, name, author):
        super().__init__(name)
        self.author = author

    def label(self):
        return f"{super().label()} / {self.author}"

class ReportFormatter:
    def format_count(self, count):
        return f"총 {count}건"

class EventReport:
    def __init__(self, events, formatter):
        self.events = list(events)
        self.formatter = formatter

    def summary(self):
        return self.formatter.format_count(len(self.events))

named_book = NamedBook("Python 기초", "교육팀")
event_report = EventReport([1, 2, 3], ReportFormatter())
assert isinstance(named_book, NamedItem)
assert named_book.label() == "Python 기초 / 교육팀"
assert event_report.summary() == "총 3건"
print(named_book.label(), event_report.summary())

## 11. 미니 실습: 보안 이벤트 객체와 배치

In [ ]:
@dataclass(frozen=True)
class SecurityEvent:
    action: str
    ip: str
    port: int

    def __post_init__(self):
        if not isinstance(self.action, str):
            raise TypeError("action은 문자열이어야 합니다")
        action = self.action.strip().upper()
        if action not in {"ALLOW", "DENY"}:
            raise ValueError("action은 ALLOW 또는 DENY여야 합니다")
        if not isinstance(self.ip, str):
            raise TypeError("ip는 문자열이어야 합니다")
        ip = self.ip.strip()
        if not ip:
            raise ValueError("ip는 비어 있을 수 없습니다")
        if type(self.port) is not int:
            raise TypeError("port는 정수여야 합니다")
        if not 1 <= self.port <= 65535:
            raise ValueError("port는 1~65535 범위여야 합니다")
        object.__setattr__(self, "action", action)
        object.__setattr__(self, "ip", ip)

    @classmethod
    def from_mapping(cls, data):
        if not isinstance(data, dict):
            raise TypeError("data는 딕셔너리여야 합니다")
        required = {"action", "ip", "port"}
        missing = required - data.keys()
        if missing:
            raise ValueError(f"필수 필드 누락: {sorted(missing)}")
        return cls(data["action"], data["ip"], data["port"])

    def endpoint(self):
        return f"{self.ip}:{self.port}"

@dataclass
class EventBatch:
    events: list[SecurityEvent] = field(default_factory=list)

    def add(self, event):
        if not isinstance(event, SecurityEvent):
            raise TypeError("SecurityEvent만 추가할 수 있습니다")
        self.events.append(event)

    def count_by_action(self):
        counts = {"ALLOW": 0, "DENY": 0}
        for event in self.events:
            counts[event.action] += 1
        return counts

    def count_by_port(self):
        counts = {}
        for event in self.events:
            counts[event.port] = counts.get(event.port, 0) + 1
        return counts

    def find_by_action(self, action):
        normalized = action.strip().upper()
        if normalized not in {"ALLOW", "DENY"}:
            raise ValueError("action은 ALLOW 또는 DENY여야 합니다")
        return [event for event in self.events if event.action == normalized]

In [ ]:
parsed_events = [
    {"action": "allow", "ip": "192.0.2.10", "port": 443},
    {"action": "DENY", "ip": "198.51.100.4", "port": 22},
    {"action": "ALLOW", "ip": "203.0.113.8", "port": 443},
]

batch = EventBatch()
for event_data in parsed_events:
    batch.add(SecurityEvent.from_mapping(event_data))

assert batch.count_by_action() == {"ALLOW": 2, "DENY": 1}
assert batch.count_by_port() == {443: 2, 22: 1}
assert [event.endpoint() for event in batch.find_by_action("allow")] == [
    "192.0.2.10:443",
    "203.0.113.8:443",
]
assert batch.events[0] == SecurityEvent("ALLOW", "192.0.2.10", 443)
expect_exception(TypeError, batch.add, {"action": "ALLOW"})
expect_exception(ValueError, SecurityEvent, "BLOCK", "192.0.2.1", 80)
expect_exception(ValueError, SecurityEvent.from_mapping, {"action": "ALLOW"})
print(batch.count_by_action(), batch.count_by_port())

## 12. 객체 컬렉션과 최종 확인

In [ ]:
ordered_events = sorted(batch.events, key=lambda event: (event.port, event.ip))
assert [event.port for event in ordered_events] == [22, 443, 443]
assert all(isinstance(event, SecurityEvent) for event in batch.events)
assert EventBatch().events == []
assert EventBatch().events is not EventBatch().events
print("전체 실습 통과!", ordered_events)

다음 질문에 답할 수 있으면 완료입니다.

1. 인스턴스 변수와 클래스 변수는 언제 각각 사용하나요?
2. 변경 가능한 클래스 변수가 왜 위험한가요?
3. `obj.method()`에서 `self`는 어떻게 전달되나요?
4. `default_factory`와 `__post_init__`의 역할은 무엇인가요?
5. `frozen=True`가 깊은 불변성을 보장하지 않는 이유는 무엇인가요?
6. 상속보다 합성이 자연스러운 관계의 예를 설명할 수 있나요?